# ⚡ RelayVSR T4 Studio — Video & Image Super-Resolution (2× & 4×)
Notebook peluncur mandiri untuk **RelayVSR** (`Wan2.2-5B Sparse Keyframe Relay` + `31.9M Dual-Memory Flash Transformer`) pada **Google Colab Tesla T4 (15 GB VRAM / 12.7 GB System RAM)**.
- **Sel 1**: Clone repositori dari GitHub (mendukung branch kustom), kompilasi *Native C++ VFS Page-Cache Cleaner*, pasang custom node `Comfyui-Relay-VSR`, dan unduh bobot `INT8 ConvRot` (`5.29 GB`).
- **Sel 2**: Luncurkan antarmuka Gradio Studio untuk Upscaling Gambar (Single/Batch) & Streaming Video.

In [ ]:
# @title 1. Setup Sistem, Git Branch & Pilih Bundle Model RelayVSR (Bootstrap T4)
import os, re, sys, shutil, subprocess
from pathlib import Path

# @markdown ### 📦 Sumber Repositori GitHub (Mendukung Nama Branch & Link `/tree/<branch>`)
REPO_URL = "https://github.com/your-username/relayvsr-t4-studio.git" # @param {type:"string"}
REPO_BRANCH = "main" # @param {type:"string"}
FORCE_GIT_PULL = True # @param {type:"boolean"}

# @markdown ---
# @markdown ### ☑️ Pilih Bundle RelayVSR Awal yang Ingin Diunduh Sekarang
# @markdown *(Rekomendasi T4: Aktifkan `INSTALL_RELAYVSR_X2_INT8` atau `INSTALL_RELAYVSR_X4_INT8` — hanya ~5.29 GB & mendukung Tensor Core INT8 Turing sm_75)*
INSTALL_RELAYVSR_X2_INT8 = True # @param {type:"boolean"}
INSTALL_RELAYVSR_X4_INT8 = False # @param {type:"boolean"}
INSTALL_RELAYVSR_X2_FP16 = False # @param {type:"boolean"}
INSTALL_RELAYVSR_X4_FP16 = False # @param {type:"boolean"}

# @markdown ---
# @markdown ### 🔑 Token Hugging Face Opsional
HF_TOKEN = "" # @param {type:"string"}
os.environ["HF_TOKEN"] = HF_TOKEN.strip()

def resolve_repo_and_branch(raw_url: str, raw_branch: str):
    url = (raw_url or "").strip()
    branch = (raw_branch or "main").strip() or "main"
    m = re.match(r"^(https?://github\.com/[^/]+/[^/#?]+?)(?:\.git)?/tree/([^/?#]+).*$", url)
    if m:
        url = m.group(1) + ".git"
        if branch == "main":
            branch = m.group(2)
    return url, branch

CLEAN_REPO_URL, TARGET_BRANCH = resolve_repo_and_branch(REPO_URL, REPO_BRANCH)
STUDIO_DIR = Path("/content/relayvsr-t4-studio") if Path("/content").exists() else Path.cwd()

# 1. Clone atau sinkronisasi branch GitHub
if (STUDIO_DIR / ".git").exists():
    if FORCE_GIT_PULL:
        print(f"🔄 Menyinkronkan {STUDIO_DIR} ke branch '{TARGET_BRANCH}'...")
        subprocess.run(["git", "-C", str(STUDIO_DIR), "fetch", "origin", TARGET_BRANCH], check=False)
        subprocess.run(["git", "-C", str(STUDIO_DIR), "checkout", "-B", TARGET_BRANCH, f"origin/{TARGET_BRANCH}"], check=False)
        subprocess.run(["git", "-C", str(STUDIO_DIR), "pull", "--ff-only", "origin", TARGET_BRANCH], check=False)
elif not (STUDIO_DIR / "bootstrap.py").exists():
    if Path("bootstrap.py").exists():
        STUDIO_DIR = Path.cwd()
    else:
        print(f"📥 Meng-clone {CLEAN_REPO_URL} (branch: {TARGET_BRANCH}) ke {STUDIO_DIR}...")
        subprocess.run(
            ["git", "clone", "--depth", "1", "--branch", TARGET_BRANCH, "--single-branch", CLEAN_REPO_URL, str(STUDIO_DIR)],
            check=True,
        )

if str(STUDIO_DIR) not in sys.path:
    sys.path.insert(0, str(STUDIO_DIR))

# 2. Susun daftar bundle yang dicentang
selected_bundles = []
if INSTALL_RELAYVSR_X2_INT8: selected_bundles.append("relayvsr_x2_int8")
if INSTALL_RELAYVSR_X4_INT8: selected_bundles.append("relayvsr_x4_int8")
if INSTALL_RELAYVSR_X2_FP16: selected_bundles.append("relayvsr_x2_fp16")
if INSTALL_RELAYVSR_X4_FP16: selected_bundles.append("relayvsr_x4_fp16")

cmd = [
    sys.executable,
    "-u",
    str(STUDIO_DIR / "bootstrap.py"),
    "--engines",
    ",".join(selected_bundles) if selected_bundles else "none",
]
if HF_TOKEN.strip():
    cmd.extend(["--hf-token", HF_TOKEN.strip()])

with subprocess.Popen(
    cmd,
    cwd=str(STUDIO_DIR),
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
) as proc:
    for line in proc.stdout:
        print(line, end="", flush=True)
    retcode = proc.wait()
    if retcode != 0:
        raise RuntimeError(f"Bootstrap gagal dengan kode keluar {retcode}. Periksa log di atas.")


In [ ]:
# @title 2. Jalankan RelayVSR T4 Studio UI (Gradio Public Link)
import subprocess, sys
from pathlib import Path

STUDIO_DIR = Path("/content/relayvsr-t4-studio") if Path("/content/relayvsr-t4-studio").exists() else Path.cwd()
PORT = 7860 # @param {type:"integer"}

cmd = [sys.executable, "-u", str(STUDIO_DIR / "run_studio.py"), "--port", str(PORT), "--share"]
with subprocess.Popen(
    cmd,
    cwd=str(STUDIO_DIR),
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
) as proc:
    for line in proc.stdout:
        print(line, end="", flush=True)
